In [ ]:
#Bellinzona 

In [ ]:
#### Morges precipitation with background

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Rectangle
from pathlib import Path
import rasterio

import requests
from PIL import Image
from io import BytesIO

# --------------------------------------------------
# INPUTS
# --------------------------------------------------
nc_file = "/storage/homefs/ge24z347/Zell_event/Data_forprocess/Combiprecip/cpch_bellinzona_giubiasco_ti_2021_2021-08-05-2021-08-07.nc"
dem_file = "/storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Bellinzona_2m_accv2/Bellinzona_2m_accv2.dem"

var_name = "RR"
time_coord = "time"

out_dir = Path("/storage/homefs/ge24z347/Zell_event/BELLINZONA_PLOTS/")
out_dir.mkdir(exist_ok=True)

# QGIS extent (EPSG:2056)
plot_extent = (2710343.040, 2741804.757, 1111963.026, 1138114.749)   # xmin, xmax, ymin, ymax
xmin, xmax, ymin, ymax = plot_extent

# Exact times
times_exact = np.array([
    "2021-08-07T12:00:00",
    "2021-08-07T13:00:00",
    "2021-08-07T14:00:00",
    "2021-08-07T15:00:00",
    "2021-08-07T16:00:00",
    "2021-08-07T17:00:00",
    "2021-08-07T18:00:00",
    "2021-08-07T19:00:00",
    "2021-08-07T20:00:00",
    "2021-08-07T21:00:00",
    "2021-08-07T22:00:00",
    "2021-08-07T23:00:00",
], dtype="datetime64[ns]")

# Swisstopo WMS layer
wms_layer = "ch.swisstopo.swisstlm3d-karte-grau"

precip_alpha = 0.70
wms_start_resolution_m = 2
wms_max_pixels = 40_000_000
save_dpi = 400

# --------------------------------------------------
# Robust WMS fetch (auto-coarsen if image would be huge)
# --------------------------------------------------
def get_swisstopo_background_image(
    xmin, xmax, ymin, ymax,
    resolution_m=2,
    layer="ch.swisstopo.swisstlm3d-karte-grau",
    max_pixels=10_000_000
):
    xmin, xmax = float(min(xmin, xmax)), float(max(xmin, xmax))
    ymin, ymax = float(min(ymin, ymax)), float(max(ymin, ymax))
    dx, dy = xmax - xmin, ymax - ymin

    res = float(resolution_m)
    while True:
        width_px = int(np.ceil(dx / res))
        height_px = int(np.ceil(dy / res))
        if width_px * height_px <= max_pixels:
            break
        res *= 2

    bbox = f"{xmin},{ymin},{xmax},{ymax}"
    params = {
        "SERVICE": "WMS",
        "REQUEST": "GetMap",
        "VERSION": "1.3.0",
        "LAYERS": layer,
        "BBOX": bbox,
        "CRS": "EPSG:2056",
        "WIDTH": width_px,
        "HEIGHT": height_px,
        "FORMAT": "image/png",
        "TRANSPARENT": "TRUE",
    }
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "image/png,image/*,*/*;q=0.8"
    }

    r = requests.get("https://wms.geo.admin.ch/", params=params, headers=headers, timeout=60)
    if r.status_code != 200:
        print("Failed to fetch WMS:", r.status_code)
        return None, res

    ctype = r.headers.get("Content-Type", "")
    if "image" not in ctype.lower():
        print("WMS returned non-image content:", ctype)
        print(r.text[:250])
        return None, res

    return Image.open(BytesIO(r.content)).convert("RGBA"), res


# --------------------------------------------------
# LOAD DATA
# --------------------------------------------------
ds = xr.open_dataset(nc_file, decode_times=True)

print("Coordinate uniqueness:")
print(f"  time unique: {ds.indexes[time_coord].is_unique}")
print(f"  x unique:    {ds.indexes['x'].is_unique}")
print(f"  y unique:    {ds.indexes['y'].is_unique}")

# Remove duplicated time values if needed
if not ds.indexes[time_coord].is_unique:
    _, unique_idx = np.unique(ds[time_coord].values, return_index=True)
    ds = ds.isel({time_coord: np.sort(unique_idx)})
    print(f"Removed duplicate time entries. New time length: {ds.sizes[time_coord]}")

da = ds[var_name]

# --------------------------------------------------
# READ DEM BOUNDS
# --------------------------------------------------
with rasterio.open(dem_file) as src:
    dem_left, dem_bottom, dem_right, dem_top = src.bounds

print("DEM bounds:")
print(f"left={dem_left}, right={dem_right}, bottom={dem_bottom}, top={dem_top}")

# --------------------------------------------------
# SPATIAL SUBSET
# --------------------------------------------------
x_ascending = bool(da.x.values[0] < da.x.values[-1])
y_ascending = bool(da.y.values[0] < da.y.values[-1])

x_slice = slice(xmin, xmax) if x_ascending else slice(xmax, xmin)
y_slice = slice(ymin, ymax) if y_ascending else slice(ymax, ymin)

da_space = da.sel(x=x_slice, y=y_slice)

print("Subset shape after spatial selection:", da_space.shape)

# --------------------------------------------------
# TIME SUBSET (safe selection)
# --------------------------------------------------
available_times = da_space[time_coord].values
common_times = np.intersect1d(available_times, times_exact)

if len(common_times) == 0:
    raise ValueError("None of the requested times were found in the dataset.")

missing_times = np.setdiff1d(times_exact, available_times)
if len(missing_times) > 0:
    print("These requested times were NOT found and will be skipped:")
    print(missing_times)

da_sel = da_space.sel({time_coord: common_times})

print("Selected times:")
print(da_sel[time_coord].values)

# --------------------------------------------------
# DISCRETE "RADAR-LIKE" LEVELS
# --------------------------------------------------
levels_full = np.array(
    [0.1, 0.2, 0.5, 1, 2, 3, 5, 7, 10, 15, 20, 30, 40, 50, 60, 80, 100, 130, 160, 250, 350],
    dtype=float
)

vmax = float(np.nanmax(da_sel.values))
if not np.isfinite(vmax) or vmax <= 0:
    raise ValueError("No positive precipitation values in the selected extent/times.")

levels = levels_full[levels_full <= max(1.0, vmax)]
if len(levels) < 2:
    levels = np.array([0.1, max(1.0, vmax)])

if levels[-1] < vmax:
    higher_levels = levels_full[levels_full > levels[-1]]
    if len(higher_levels) > 0:
        levels = np.append(levels, higher_levels[0])
    else:
        levels = np.append(levels, vmax)

mswiss_15 = [
    "#F9844A", "#FBB476", "#FDD8A3",
    "#FFFFCC", "#F0E68C",
    "#C7E9B4", "#7FCDBB",
    "#41B6C4", "#1D91C0",
    "#225EA8", "#253494",
    "#081D58", "#31183b",
    "#70265c", "#bc3754"
]

n_bins = len(levels) - 1
base = plt.get_cmap(ListedColormap(mswiss_15))
colors = base(np.linspace(0, 1, n_bins))
cmap = ListedColormap(colors)
cmap.set_bad((0, 0, 0, 0))
norm = BoundaryNorm(levels, ncolors=cmap.N, clip=True)

print(f"vmax in window: {vmax:.2f} mm")
print("levels used:", levels)

# --------------------------------------------------
# GET BACKGROUND ONCE
# --------------------------------------------------
bg_img, used_res = get_swisstopo_background_image(
    xmin, xmax, ymin, ymax,
    resolution_m=wms_start_resolution_m,
    layer=wms_layer,
    max_pixels=wms_max_pixels
)
print(f"WMS resolution used: {used_res} m/px")

# --------------------------------------------------
# LUMINO LOCATION
# --------------------------------------------------
morges_x = 2725399.448
morges_y = 1121160.463

# --------------------------------------------------
# PLOT
# --------------------------------------------------
for t in da_sel[time_coord].values:
    frame = da_sel.sel({time_coord: t}).astype(float)
    frame = frame.where(frame > 0)

    fig, ax = plt.subplots(figsize=(8, 7))
    fig.patch.set_alpha(0)
    ax.set_facecolor("none")

    if bg_img is not None:
        ax.imshow(bg_img, extent=(xmin, xmax, ymin, ymax), origin="upper", zorder=0)
    else:
        print("No WMS background for", t)

    im = ax.imshow(
        frame.values,
        extent=(xmin, xmax, ymin, ymax),
        origin="lower",
        interpolation="nearest",
        cmap=cmap,
        norm=norm,
        alpha=precip_alpha,
        zorder=2
    )

    # DEM rectangle
    rect = Rectangle(
        (dem_left, dem_bottom),
        dem_right - dem_left,
        dem_top - dem_bottom,
        fill=False,
        edgecolor="red",
        linewidth=2.0,
        zorder=5
    )
    ax.add_patch(rect)

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_aspect("equal")
    ax.set_title(str(np.datetime64(t))[:16], color="black")

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, ticks=levels)
    cbar.set_label("precipitation (mm)")
    cbar.ax.tick_params(colors="black")
    cbar.outline.set_edgecolor("black")

    ax.scatter(
        morges_x,
        morges_y,
        marker="^",
        s=50,
        edgecolor="black",
        facecolor="yellow",
        linewidth=1.5,
        zorder=6
    )

    ax.text(
        morges_x + 500,
        morges_y + 500,
        "Lumino",
        fontsize=8,
        color="black",
        weight="bold",
        zorder=6
    )

    out = out_dir / f"CPC_BELLINZONA_{str(t)[:19].replace(':','')}.png"
    plt.savefig(out, dpi=save_dpi, transparent=True, bbox_inches="tight")
    plt.close()

print("Done.")

In [ ]:
### Review the data of the dem of it have nan 

In [ ]:
import rasterio
import numpy as np

dem_file = "/storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Bellinzona_2m_accv2/Bellinzona_2m_accv2.dem"

with rasterio.open(dem_file) as src:
    dem = src.read(1)
    nodata = src.nodata
    bounds = src.bounds
    transform = src.transform
    crs = src.crs

print("shape:", dem.shape)
print("dtype:", dem.dtype)
print("crs:", crs)
print("bounds:", bounds)
print("nodata metadata:", nodata)

print("Any NaN?:", np.isnan(dem).any())
print("NaN count:", np.isnan(dem).sum())

if nodata is not None:
    print("Count equal to nodata metadata:", np.sum(dem == nodata))

finite_mask = np.isfinite(dem)
print("Finite cells:", finite_mask.sum())
print("Non-finite cells:", (~finite_mask).sum())

if finite_mask.any():
    print("Min finite value:", np.nanmin(dem))
    print("Max finite value:", np.nanmax(dem))